# E-QUICK — three cheap controls, no training, no generation

| # | Control | Why | Cost |
|---|---|---|---|
| Q1 | **Recover `A_raw_s0_r16`'s ‖lora_B‖** from its safetensors | that adapter predates the norm check, so arm A's covariate analysis runs at n=5 with a hole in the table | seconds |
| Q2 | **Multiple spectrally-matched realisations** | `ctrl_Q` vs K_Q came in at t = −3.63, but its comparator is K_A whose generic offset need not match K_Q's. One Gaussian realisation cannot establish a PRNU-vs-matched-noise difference (review §6). Measure the **existing** ctrl_Q generations against several independent fields and see whether K_Q sits inside their spread | ~5 min |
| Q3 | **E5 — second fingerprint extractor** | kills "your null is an artifact of the Mihcak denoiser". Re-measures the arms that carry the bound with a different wavelet basis and level | ~25 min |

Everything reads artefacts already on Drive. GPU optional (used only to speed up residuals).

**Run all.**

In [ ]:
# %% Q0 — setup
import os, sys, json, glob, math, time, gc, hashlib, subprocess
from concurrent.futures import ThreadPoolExecutor
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE="/content/drive/MyDrive"
except Exception:
    DRIVE=os.environ.get("EINV_DRIVE",".")
BASE_ROOT=os.path.join(DRIVE,"inv_channel/E_INV_P0_v3")
SEED_ROOT=os.path.join(DRIVE,"inv_channel/E_SEEDEXT")
AMP_ROOT =os.path.join(DRIVE,"inv_channel/E_AMP")
OUT=os.path.join(BASE_ROOT,"quick"); os.makedirs(OUT,exist_ok=True)

subprocess.run([sys.executable,"-m","pip","install","-q","PyWavelets","safetensors"],check=False)
import numpy as np, pandas as pd, pywt, torch
import torch.nn.functional as tF
from PIL import Image
from scipy import stats as sps
DEV="cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.allow_tf32=False; torch.backends.cuda.matmul.allow_tf32=False
np.seterr(divide="raise",invalid="raise")
MEAS=1024; _MEASURE_DEV=DEV
print("device:",DEV,"| MEAS:",MEAS)

def load_lum_crop(fp):
    with Image.open(fp) as im:
        im=im.convert("RGB"); W,H=im.size
        a=np.asarray(im.crop(((W-MEAS)//2,(H-MEAS)//2,(W-MEAS)//2+MEAS,(H-MEAS)//2+MEAS)),np.float32)
    return (0.299*a[...,0]+0.587*a[...,1]+0.114*a[...,2]).astype(np.float32)

def _conv2(x,k):
    xt=torch.from_numpy(np.ascontiguousarray(x))[None,None]; kt=torch.from_numpy(k)[None,None]
    if _MEASURE_DEV=="cuda":
        return tF.conv2d(xt.cuda(),kt.cuda(),padding=k.shape[0]//2)[0,0].cpu().numpy()
    return tF.conv2d(xt,kt,padding=k.shape[0]//2)[0,0].numpy()

def wavelet_residual(img, wavelet="db8", levels=4, sigma0=2.0):
    s0=np.float32(sigma0)**2
    co=pywt.wavedec2(img,wavelet,level=levels,mode="periodization")
    out=[co[0]*0.0]
    for (cH,cV,cD) in co[1:]:
        band=[]
        for ch in (cH,cV,cD):
            ch=ch.astype(np.float32); vmin=None
            for w in (3,5,7,9):
                v=np.maximum(_conv2(ch*ch,np.ones((w,w),np.float32)/(w*w))-s0,0.0)
                vmin=v if vmin is None else np.minimum(vmin,v)
            band.append(ch*(s0/(vmin+s0)))
        out.append(tuple(band))
    n=pywt.waverec2(out,wavelet,mode="periodization")[:img.shape[0],:img.shape[1]]
    r=n-n.mean(); sd=float(r.std())
    if not np.isfinite(sd) or sd==0.0: raise SystemExit("degenerate residual")
    return (r/sd).astype(np.float32)

def ncc0(a,b):
    a=a-a.mean(); b=b-b.mean()
    na=float(np.linalg.norm(a)); nb=float(np.linalg.norm(b))
    return 0.0 if na==0 or nb==0 else float((a*b).sum()/(na*nb))
def _ncc_batch(a,B):
    a=a-a.mean(); na=a.norm()
    Bc=B-B.mean(dim=(1,2),keepdim=True); nb=Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1)/(na*nb).clamp_min(1e-30)
def append_row(p,h,r):
    new=not os.path.exists(p)
    with open(p,"a") as f:
        if new: f.write(",".join(h)+"\n")
        f.write(",".join(str(x) for x in r)+"\n")
def done_keys(p,cols):
    if not os.path.exists(p): return set()
    import csv as _c
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _c.DictReader(f)}

In [ ]:
# %% Q1 — recover the missing ||lora_B|| and rebuild the covariate table at full n
from safetensors.torch import load_file

def norm_of(root,tag):
    mp=os.path.join(root,"adapters",tag,"train_meta.json")
    m=json.load(open(mp)) if os.path.exists(mp) else {}
    b=m.get("lora_B_norm")
    if b is None:
        sd=load_file(os.path.join(root,"adapters",tag,"pytorch_lora_weights.safetensors"))
        b=float(sum(float(v.float().norm()) for k,v in sd.items() if "lora_B" in k))
        m["lora_B_norm"]=b; m["lora_B_norm_recovered"]=True
        tmp=mp+".tmp"; json.dump(m,open(tmp,"w"),indent=2); os.replace(tmp,mp)
        print(f"  recovered ||lora_B||={b:.3f} for {tag} and wrote it back")
    return b, m.get("loss_tail")

BASE=pd.read_csv(os.path.join(BASE_ROOT,"csv","s5_measure.csv"))
EXT =pd.read_csv(os.path.join(SEED_ROOT,"csv","b3_measure.csv"))
def theta_of(df,tag,ks,ko):
    a=df[(df.tag==tag)&(df.K==ks)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    b=df[(df.tag==tag)&(df.K==ko)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    n=min(len(a),len(b)); return (a[:n]-b[:n]).mean()

rows=[]
for arm,ks,ko in (("A","A","B"),("B","B","A")):
    for s,root,df in [(s,BASE_ROOT,BASE) for s in (0,1,2)]+[(s,SEED_ROOT,EXT) for s in (3,4,5)]:
        t=f"{arm}_raw_s{s}_r16"; b,l=norm_of(root,t)
        rows.append((arm,t,theta_of(df,t,ks,ko),b,l))
cov=pd.DataFrame(rows,columns=["arm","tag","theta","lora_B","tail"])
print(cov.to_string(index=False,float_format=lambda x:f"{x:.5g}"))
print()
for arm in ("A","B"):
    g=cov[cov.arm==arm]
    pr,pp=sps.pearsonr(g.theta,g.lora_B); sr,sp=sps.spearmanr(g.theta,g.lora_B)
    pr2,pp2=sps.pearsonr(g.theta,g["tail"])
    print(f"  arm {arm} (n={len(g)}): theta~||lora_B|| Pearson {pr:+.3f} (p={pp:.3f}) "
          f"Spearman {sr:+.3f} (p={sp:.3f}) | theta~tail Pearson {pr2:+.3f} (p={pp2:.3f})")
cov.to_csv(os.path.join(OUT,"covariates.csv"),index=False)
print("\n-> quick/covariates.csv   (EXPLORATORY: report both correlation measures)")

In [ ]:
# %% Q2 — multiple spectrally-matched realisations
#     Measures the EXISTING ctrl_Q generations against several independent Gaussian fields with
#     the same power spectrum as K_B. If K_Q sits inside that spread, the t=-3.63 was comparator
#     bias, not a matched-noise effect.
N_REAL   = 4          # independent Q realisations, seeds 1..4 (seed 0 is the injected one)
N_IMGS   = 500
def spectral_match(K, seed):
    rng=np.random.default_rng(seed)
    mag=np.abs(np.fft.fft2(K.astype(np.float32)))
    ph=np.fft.fft2(rng.standard_normal(K.shape).astype(np.float32)); ph=ph/np.maximum(np.abs(ph),1e-12)
    Q=np.real(np.fft.ifft2(mag*ph)).astype(np.float32); Q=Q-Q.mean()
    return (Q*(K.std()/max(Q.std(),1e-12))).astype(np.float32)

KB_E1=np.load(os.path.join(BASE_ROOT,"fingerprints","K_B_E1.npy"))
KQ   =np.load(os.path.join(AMP_ROOT,"fingerprints","K_Q.npy"))       # the injected realisation
KA   =np.load(os.path.join(AMP_ROOT,"fingerprints","K_A.npy"))
fields={"K_Q (injected)":KQ, "K_A":KA}
for i in range(1,N_REAL+1): fields[f"Q_alt{i}"]=spectral_match(KB_E1,seed=i)
names=list(fields)
print("field cross-correlations (all should be ~0 except with itself):")
for n_ in names[2:]:
    print(f"  NCC({n_}, K_Q)={ncc0(fields[n_],KQ):+.5f}   sd ratio {fields[n_].std()/KQ.std():.4f}")

STK=torch.stack([torch.from_numpy(fields[n_]) for n_ in names]).to(_MEASURE_DEV)
paths=sorted(glob.glob(os.path.join(AMP_ROOT,"gens","ctrl_Q","*.png")))[:N_IMGS]
csvq=os.path.join(OUT,"q_realisations.csv"); done=done_keys(csvq,["idx"])
ex=ThreadPoolExecutor(max_workers=4)
try:
    todo=[i for i in range(len(paths)) if str(i) not in {d[0] for d in done}]
    fut={i:ex.submit(load_lum_crop,paths[i]) for i in todo[:6]}
    t0=time.time()
    for pos,i in enumerate(todo):
        Z_np=fut.pop(i).result(); nx=pos+6
        if nx<len(todo): fut[todo[nx]]=ex.submit(load_lum_crop,paths[todo[nx]])
        W=torch.from_numpy(wavelet_residual(Z_np)).to(_MEASURE_DEV)
        Z=torch.from_numpy(Z_np).to(_MEASURE_DEV)
        v=_ncc_batch(W,Z.unsqueeze(0)*STK).cpu().numpy()
        append_row(csvq,["idx"]+[n_.replace(" ","_") for n_ in names],[i]+[f"{x:.6e}" for x in v])
    print(f"[Q2] measured {len(todo)} generations ({time.time()-t0:.0f}s)")
finally:
    ex.shutdown(wait=True); gc.collect()
    if DEV=="cuda": torch.cuda.empty_cache()

d=pd.read_csv(csvq); ref=d["K_A"].to_numpy(float)
print(f"\n{'field':18s} {'mean rho':>12s} {'contrast vs K_A':>16s} {'t':>7s}")
alt=[]
for n_ in names:
    col=d[n_.replace(" ","_")].to_numpy(float)
    ctr=col-ref; se=ctr.std(ddof=1)/np.sqrt(len(ctr))
    t_=ctr.mean()/se if se else np.nan
    print(f"{n_:18s} {col.mean():12.3e} {ctr.mean():+16.3e} {t_:7.2f}")
    if n_.startswith("Q_alt"): alt.append(ctr.mean())
alt=np.array(alt); kq=(d["K_Q_(injected)"].to_numpy(float)-ref).mean()
print(f"\n  alternative realisations: mean {alt.mean():+.3e}  sd {alt.std(ddof=1):.3e}  "
      f"range [{alt.min():+.3e}, {alt.max():+.3e}]")
z=(kq-alt.mean())/alt.std(ddof=1) if len(alt)>1 else np.nan
print(f"  injected K_Q: {kq:+.3e}   z vs the alternatives = {z:+.2f}")
print("  READ: |z| small -> the negative was COMPARATOR BIAS, and no matched-noise transfer is")
print("        indicated. |z| large and negative -> something specific to the injected field.")

In [ ]:
# %% Q3 — E5: second fingerprint extractor on the arms that carry the bound
EXTRACTORS={"db8_L4 (primary)":dict(wavelet="db8",levels=4,sigma0=2.0),
            "sym8_L3 (alt)":   dict(wavelet="sym8",levels=3,sigma0=3.0)}
TAGS=[(f"{a}_raw_s{s}_r16", BASE_ROOT if s<3 else SEED_ROOT, a) for a in ("A","B") for s in range(6)]
N5=250        # per tag; the contrast is a mean so this is ample for a robustness check

K={r:torch.from_numpy(np.load(os.path.join(BASE_ROOT,"fingerprints",f"K_{r}_E2.npy"))).to(_MEASURE_DEV)
   for r in ("A","B")}
KS=torch.stack([K["A"],K["B"]])
csv5=os.path.join(OUT,"e5_extractors.csv"); done5=done_keys(csv5,["extractor","tag","idx"])
for ename,kw in EXTRACTORS.items():
    for tag,root,arm in TAGS:
        gd=os.path.join(root if root==SEED_ROOT else BASE_ROOT,"gens",tag)
        paths=sorted(glob.glob(os.path.join(gd,"*.png")))[:N5]
        todo=[i for i in range(len(paths)) if (ename,tag,str(i)) not in done5]
        if not todo: continue
        t0=time.time()
        for i in todo:
            Z_np=load_lum_crop(paths[i])
            W=torch.from_numpy(wavelet_residual(Z_np,**kw)).to(_MEASURE_DEV)
            Z=torch.from_numpy(Z_np).to(_MEASURE_DEV)
            v=_ncc_batch(W,Z.unsqueeze(0)*KS).cpu().numpy()
            append_row(csv5,["extractor","tag","arm","idx","rho_A","rho_B"],
                       [ename,tag,arm,i,f"{v[0]:.6e}",f"{v[1]:.6e}"])
        print(f"[Q3] {ename:18s} {tag:16s} {len(todo)} imgs ({time.time()-t0:.0f}s)")

d5=pd.read_csv(csv5)
print(f"\n{'extractor':20s} {'arm':4s} {'n_ad':>5s} {'theta':>12s} {'sd':>11s} {'U(99% sim)':>12s}")
res5={}
for ename in EXTRACTORS:
    for arm in ("A","B"):
        g=d5[(d5.extractor==ename)&(d5.arm==arm)]
        means=[]
        for tag,_,a in [t for t in TAGS if t[2]==arm]:
            h=g[g.tag==tag]
            if not len(h): continue
            th=(h.rho_A-h.rho_B) if arm=="A" else (h.rho_B-h.rho_A)
            means.append(float(th.mean()))
        if len(means)<3: continue
        m=np.array(means); k=len(m)
        U=float(m.mean()+sps.t.ppf(0.995,df=k-1)*m.std(ddof=1)/np.sqrt(k))
        res5[(ename,arm)]=U
        print(f"{ename:20s} {arm:4s} {k:5d} {m.mean():+12.3e} {m.std(ddof=1):11.3e} {U:+12.3e}")
if len(res5)>=4:
    Up={a:max(res5[(e,a)] for e in EXTRACTORS if (e,a) in res5) for a in ("A","B")}
    for ename in EXTRACTORS:
        U_=max(res5.get((ename,"A"),-9),res5.get((ename,"B"),-9))
        print(f"\n  U_device with {ename:18s} = {U_:+.3e}")
    print("\n  READ: if the two extractors give the same order of magnitude for U_device, the null")
    print("        is not an artifact of the denoiser. Report both in a robustness table.")
print("\n  NOTE: n=250/tag here (robustness check), vs 500 in the headline analysis — the bound")
print("        from this cell is therefore looser and is NOT the number to report.")

## What each cell settles

**Q1** fills the hole in the covariate table and writes the recovered norm back into
`train_meta.json`, so every later analysis sees it. The correlation stays exploratory — report
Pearson *and* Spearman, because arm A's Pearson is carried by one leverage point.

**Q2** is the review's §6 point made concrete. If the injected K_Q's contrast sits within the
spread of independent realisations (small |z|), the t = −3.63 was comparator bias and there is no
matched-noise effect to explain. Only then may you write *"nor did the tested spectrally matched
Gaussian control"* — and never *"nor does matched noise"*.

**Q3** is the construct-validity check. What matters is whether U_device lands in the same order of
magnitude under a different wavelet basis, level count and noise floor — not whether the numbers
match exactly. Report both extractors in a robustness table; the headline bound stays the db8/L4
one computed at full n.